# 04 — Análise estatística (nível estadual — 645 municípios)

**Objetivo:** testar a hipótese central do estudo:

> Municípios de SP com maior proporção de idosos morando sozinhos têm maior
> taxa de internação por causas associadas a falta de socorro imediato
> (lesões/causas externas, sintomas mal definidos, transtornos mentais),
> mesmo controlando pelo IDH municipal?

**Entrada:** `data/processed/dataset_consolidado_sp.csv` (gerado no notebook 03)

**Saídas:** tabelas e figuras em `outputs/tables/` e `outputs/figures/` —
prontas para entrar na seção de Resultados do artigo.


In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path("..") / "src"))
import config  # noqa: E402

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.formula.api as smf

sns.set_theme(style="whitegrid")

df = pd.read_csv(config.DATA_PROCESSED / "dataset_consolidado_sp.csv")
df.head()


## 4.1 Estatística descritiva


In [ ]:
df.describe(include="all").T


## 4.2 Correlação: % idosos sozinhos × taxa de internação

(depende das colunas `pct_idosos_sozinhos` e `taxa_internacao_100k_idosos`
criadas no notebook 03, seção 3.3 — ajuste os nomes abaixo se você usou
nomes diferentes)


In [ ]:
x = "pct_idosos_sozinhos"
y = "taxa_internacao_100k_idosos"

if x in df.columns and y in df.columns:
    r, p = stats.pearsonr(df[x].dropna(), df.loc[df[x].notna(), y])
    print(f"Correlação de Pearson: r={r:.3f}, p={p:.4f}")

    fig, ax = plt.subplots(figsize=(8, 6))
    sns.regplot(data=df, x=x, y=y, ax=ax, scatter_kws={"alpha": 0.5})

    rc = df[df["codigo_ibge"] == config.RIO_CLARO_CODIGO_IBGE]
    if not rc.empty:
        ax.scatter(rc[x], rc[y], color="red", s=100, zorder=5, label=config.RIO_CLARO_NOME)
        ax.legend()

    ax.set_xlabel("% de idosos morando sozinhos")
    ax.set_ylabel("Internações por 100 mil idosos (causas do estudo)")
    ax.set_title("Idosos sozinhos × internações — municípios de SP")
    fig.tight_layout()
    fig.savefig(config.OUTPUTS_FIGURES / "correlacao_idosos_sozinhos_internacoes.png", dpi=150)
    plt.show()
else:
    print(f"Colunas '{x}' e/ou '{y}' não encontradas — finalize o notebook 03 antes.")


## 4.3 Regressão controlando por IDH

Modelo: `taxa_internacao_100k_idosos ~ pct_idosos_sozinhos + idhm`

(ajuste o nome da coluna de IDH conforme o que você trouxe do Atlas Brasil
no notebook 03)


In [ ]:
formula = "taxa_internacao_100k_idosos ~ pct_idosos_sozinhos + idhm"
try:
    modelo = smf.ols(formula, data=df).fit()
    print(modelo.summary())
    with open(config.OUTPUTS_TABLES / "regressao_ols.txt", "w") as f:
        f.write(modelo.summary().as_text())
except Exception as e:
    print("Ainda não dá para rodar a regressão:", e)
    print("Confirme se as colunas 'pct_idosos_sozinhos', 'taxa_internacao_100k_idosos' e 'idhm' existem em df.columns")


## 4.4 Perfil das internações por causa

Lembrando: cada "causa" aqui é um **capítulo da CID-10** (ver
`config.CAUSAS_SIH` para o detalhe de cada um e suas limitações), não o
subgrupo específico (quedas, fratura de fêmur etc.) do plano original.


In [ ]:
causas = list(config.CAUSAS_SIH.keys())
presentes = [c for c in causas if c in df.columns]

if presentes:
    totais = df[presentes].sum().rename(index=config.CAUSAS_SIH_LABELS)
    fig, ax = plt.subplots(figsize=(7, 5))
    totais.sort_values().plot(kind="barh", ax=ax, color="#4C72B0")
    ax.set_xlabel("Total de internações (estado de SP)")
    ax.set_title("Perfil das internações em idosos, por capítulo CID-10")
    fig.tight_layout()
    fig.savefig(config.OUTPUTS_FIGURES / "perfil_causas.png", dpi=150)
    plt.show()
else:
    print("Colunas de causa não encontradas em df — confira o notebook 03.")


## 4.5 Mapa coroplético (opcional — requer `geopandas` e o shapefile de SP)

1. Baixe: https://geoftp.ibge.gov.br/organizacao_do_territorio/malhas_territoriais/malhas_municipais/municipio_2022/UFs/SP/SP_Municipios_2022.zip
2. Salve (sem descompactar) como `data/external/sp_municipios.zip`


In [ ]:
try:
    import geopandas as gpd

    caminho_shp = config.DATA_EXTERNAL / "sp_municipios.zip"
    if caminho_shp.exists():
        gdf = gpd.read_file(f"zip://{caminho_shp}")
        gdf["codigo_ibge"] = gdf["CD_MUN"].astype(int)
        mapa = gdf.merge(df, on="codigo_ibge", how="left")

        fig, ax = plt.subplots(figsize=(9, 9))
        mapa.plot(column="taxa_internacao_100k_idosos", cmap="OrRd", legend=True, ax=ax,
                  missing_kwds={"color": "lightgrey"})
        ax.set_title("Taxa de internação por causas evitáveis em idosos — SP")
        ax.axis("off")
        fig.tight_layout()
        fig.savefig(config.OUTPUTS_FIGURES / "mapa_taxa_internacao_sp.png", dpi=150)
        plt.show()
    else:
        print(f"Baixe o shapefile conforme instruções acima e salve em {caminho_shp}")
except ImportError:
    print("geopandas não instalado. No Anaconda Prompt: conda install -c conda-forge geopandas")
